# Theory 15 — soft observability and slot stability

**Formal source:** [`../15_soft_observability_and_slot_stability.md`](../15_soft_observability_and_slot_stability.md)

This Notebook is an executable finite witness, not the general proof. Passing it supports implementation consistency only; it does not establish learned-model or real-PHM evidence.

In [ ]:
import math
import itertools
import numpy as np
np.set_printoptions(precision=6, suppress=True)


def four_way(projectors, domain_index, atol=1e-9):
    ps = [np.asarray(p, float) for p in projectors]
    dimension = ps[0].shape[0]
    summed = sum(ps)
    values, vectors = np.linalg.eigh((summed + summed.T) / 2)
    basis = vectors[:, np.isclose(values, len(ps), atol=atol)]
    shared = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    basis = vectors[:, values > atol]
    union = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    observed = ps[domain_index]
    blocks = [shared, observed - shared, union - observed, np.eye(dimension) - union]
    for projector in blocks:
        np.testing.assert_allclose(projector, projector.T, atol=1e-8)
        np.testing.assert_allclose(projector @ projector, projector, atol=1e-8)
    for index, left in enumerate(blocks):
        for right in blocks[index + 1:]:
            np.testing.assert_allclose(left @ right, 0, atol=1e-8)
    np.testing.assert_allclose(sum(blocks), np.eye(dimension), atol=1e-8)
    return blocks


def normal_pdf(x, mean, standard_deviation):
    return np.exp(-0.5 * ((x - mean) / standard_deviation) ** 2) / (
        math.sqrt(2 * math.pi) * standard_deviation
    )

In [ ]:
def soft(values, threshold, temperature):
    return 1 / (1 + np.exp(-(np.asarray(values) - threshold) / temperature))

weights = soft([0.5, 1.0, 1.5], 1.0, 0.05)
assert weights[0] < 1e-4 and weights[1] == 0.5 and weights[2] > 1 - 1e-4
temperature, perturbation = 0.2, 0.03
change = abs(soft([0.96], 1, temperature)[0] - soft([0.93], 1, temperature)[0])
bound = perturbation / (4 * temperature)
assert change <= bound + 1e-12
print({"hard_limit_weights": weights.tolist(), "change": float(change), "bound": bound})

In [ ]:
print('THEORY_DEMO_PASS::15_soft_observability_and_slot_stability')
print('evidence_level: constructive_or_numerical_witness')
print('formal_claim_supported: false')